# Preparing Quran Data for a RAG System

This tutorial is part of the [ChatQuran: A Multilingual RAG System](https://aviosit.com/projects/chatquran-multilingual-rag-system) project by [AviosIT](https://aviosit.com).

It explores how Quran source data can be inspected, validated, combined, and prepared for use in a retrieval-augmented generation (RAG) system.

The exploratory steps in this notebook are later turned into reusable Python code in the ChatQuran project.

## 1. Inspecting the Raw Quran Data

Before preparing the Quran dataset for retrieval, we first need to understand the structure of the raw source data.

The Arabic Quran text used in this tutorial is sourced from [Tanzil Quran Text](https://tanzil.net/download/).

The English and Indonesian translations are also sourced through [Tanzil's translation collection](https://tanzil.net/trans/).

See the [Tanzil Text License](https://tanzil.net/docs/Text_License) for the terms governing the use and distribution of the Arabic Quran text.

In [57]:
from pathlib import Path

In [58]:
data_dir = Path("../data/raw")
arabic_file = data_dir / "quran-uthmani.txt"

arabic_file

PosixPath('../data/raw/quran-uthmani.txt')

In [59]:
arabic_file.exists()

True

In [60]:
with arabic_file.open("r", encoding="utf-8") as file:
    for _ in range(10):
        print(file.readline().rstrip())

1|1|بِسْمِ ٱللَّهِ ٱلرَّحْمَـٰنِ ٱلرَّحِيمِ
1|2|ٱلْحَمْدُ لِلَّهِ رَبِّ ٱلْعَـٰلَمِينَ
1|3|ٱلرَّحْمَـٰنِ ٱلرَّحِيمِ
1|4|مَـٰلِكِ يَوْمِ ٱلدِّينِ
1|5|إِيَّاكَ نَعْبُدُ وَإِيَّاكَ نَسْتَعِينُ
1|6|ٱهْدِنَا ٱلصِّرَٰطَ ٱلْمُسْتَقِيمَ
1|7|صِرَٰطَ ٱلَّذِينَ أَنْعَمْتَ عَلَيْهِمْ غَيْرِ ٱلْمَغْضُوبِ عَلَيْهِمْ وَلَا ٱلضَّآلِّينَ
2|1|بِسْمِ ٱللَّهِ ٱلرَّحْمَـٰنِ ٱلرَّحِيمِ الٓمٓ
2|2|ذَٰلِكَ ٱلْكِتَـٰبُ لَا رَيْبَ ۛ فِيهِ ۛ هُدًى لِّلْمُتَّقِينَ
2|3|ٱلَّذِينَ يُؤْمِنُونَ بِٱلْغَيْبِ وَيُقِيمُونَ ٱلصَّلَوٰةَ وَمِمَّا رَزَقْنَـٰهُمْ يُنفِقُونَ


## 2. Understanding the Raw Data Format

Each record in the Quran source represents a verse and follows the pipe-delimited format: `surah|ayah|text`.

The first field identifies the surah, the second identifies the ayah, and the third contains the text.

In [61]:
with arabic_file.open("r", encoding="utf-8") as file:
    first_line = file.readline().rstrip("\n")

first_line

'1|1|بِسْمِ ٱللَّهِ ٱلرَّحْمَـٰنِ ٱلرَّحِيمِ'

In [62]:
parts = first_line.split("|")

parts

['1', '1', 'بِسْمِ ٱللَّهِ ٱلرَّحْمَـٰنِ ٱلرَّحِيمِ']

In [63]:
parts = first_line.split("|", maxsplit=2)

parts

['1', '1', 'بِسْمِ ٱللَّهِ ٱلرَّحْمَـٰنِ ٱلرَّحِيمِ']

## 3. Identifying Data and Metadata Lines

The raw file contains Quran verse records as well as metadata or comment lines.

Before parsing the dataset, we need to distinguish actual verse records from lines that should be skipped.

In [64]:
with arabic_file.open("r", encoding="utf-8") as file:
    lines = file.readlines()

len(lines)

6266

In [65]:
comment_lines = [line.rstrip("\n") for line in lines if line.startswith("#")]

len(comment_lines), comment_lines[:5]

(28,
 ['# PLEASE DO NOT REMOVE OR CHANGE THIS COPYRIGHT BLOCK',
  '#====================================================================',
  '#',
  '#  Tanzil Quran Text (Uthmani, Version 1.1)',
  '#  Copyright (C) 2007-2026 Tanzil Project'])

## 4. Checking Empty Lines

Not every non-comment line is necessarily a Quran verse record.

We also need to check whether the raw file contains empty lines before parsing the records.

In [66]:
empty_lines = [line for line in lines if not line.strip()]

len(empty_lines)

2

In [67]:
empty_lines[:5]

['\n', '\n']

## 5. Validating Quran Verse Records

After excluding metadata and empty lines, the remaining lines should represent Quran verse records.

We can validate the number of records and check whether each record follows the expected `surah|ayah|text` structure.

In [68]:
verse_lines = [
    line.rstrip("\n")
    for line in lines
    if line.strip() and not line.startswith("#")
]

len(verse_lines)

6236

In [69]:
record_lengths = [len(line.split("|", maxsplit=2)) for line in verse_lines]

set(record_lengths)

{3}

In [70]:
verse_lines[:5]

['1|1|بِسْمِ ٱللَّهِ ٱلرَّحْمَـٰنِ ٱلرَّحِيمِ',
 '1|2|ٱلْحَمْدُ لِلَّهِ رَبِّ ٱلْعَـٰلَمِينَ',
 '1|3|ٱلرَّحْمَـٰنِ ٱلرَّحِيمِ',
 '1|4|مَـٰلِكِ يَوْمِ ٱلدِّينِ',
 '1|5|إِيَّاكَ نَعْبُدُ وَإِيَّاكَ نَسْتَعِينُ']

In [71]:
verse_lines[-5:]

['114|2|مَلِكِ ٱلنَّاسِ',
 '114|3|إِلَـٰهِ ٱلنَّاسِ',
 '114|4|مِن شَرِّ ٱلْوَسْوَاسِ ٱلْخَنَّاسِ',
 '114|5|ٱلَّذِى يُوَسْوِسُ فِى صُدُورِ ٱلنَّاسِ',
 '114|6|مِنَ ٱلْجِنَّةِ وَٱلنَّاسِ']

In [72]:
invalid_records = [
    line for line in verse_lines
    if len(line.split("|", maxsplit=2)) != 3
]

len(invalid_records)

0

In [73]:
invalid_records[:5]

[]

## 6. Parsing Quran Verse Records

Each valid verse record contains three fields:

- `surah`: the surah number
- `ayah`: the ayah number
- `text`: the Quran text

We now parse these fields from the raw text records.

In [74]:
records = []

for line in verse_lines:
    surah, ayah, text = line.split("|", maxsplit=2)
    records.append({
        "surah": int(surah),
        "ayah": int(ayah),
        "text": text,
    })

In [75]:
records[0]

{'surah': 1, 'ayah': 1, 'text': 'بِسْمِ ٱللَّهِ ٱلرَّحْمَـٰنِ ٱلرَّحِيمِ'}

In [76]:
records[:3]

[{'surah': 1, 'ayah': 1, 'text': 'بِسْمِ ٱللَّهِ ٱلرَّحْمَـٰنِ ٱلرَّحِيمِ'},
 {'surah': 1, 'ayah': 2, 'text': 'ٱلْحَمْدُ لِلَّهِ رَبِّ ٱلْعَـٰلَمِينَ'},
 {'surah': 1, 'ayah': 3, 'text': 'ٱلرَّحْمَـٰنِ ٱلرَّحِيمِ'}]

In [77]:
type(records[0]["surah"]), type(records[0]["ayah"]), type(records[0]["text"])

(int, int, str)

## 7. Creating a DataFrame

The parsed records are now represented as Python dictionaries.

A pandas DataFrame gives us a tabular structure that is easier to inspect, validate, transform, and combine with other Quran data sources.

In [78]:
import pandas as pd

In [79]:
arabic_df = pd.DataFrame(records)

arabic_df.head()

,surah,ayah,text
0,1,1,بِسْمِ ٱللَّهِ ٱلرَّحْمَـٰنِ ٱلرَّحِيمِ
1,1,2,ٱلْحَمْدُ لِلَّهِ رَبِّ ٱلْعَـٰلَمِينَ
2,1,3,ٱلرَّحْمَـٰنِ ٱلرَّحِيمِ
3,1,4,مَـٰلِكِ يَوْمِ ٱلدِّينِ
4,1,5,إِيَّاكَ نَعْبُدُ وَإِيَّاكَ نَسْتَعِينُ


## 8. Inspecting the DataFrame

Now that the Quran records are represented as a DataFrame, we can inspect its structure and data types before combining it with other sources.

In [80]:
arabic_df.shape

(6236, 3)

In [81]:
arabic_df.columns

Index(['surah', 'ayah', 'text'], dtype='str')

In [82]:
arabic_df.dtypes

surah    int64
ayah     int64
text       str
dtype: object

## 9. Checking the Quran Verse Key

A Quran verse can be uniquely identified by the combination of its surah and ayah numbers.

We use `(surah, ayah)` as the natural key for aligning the Arabic text with its translations.

In [83]:
arabic_df.duplicated(subset=["surah", "ayah"]).sum()

np.int64(0)

In [84]:
arabic_df[["surah", "ayah"]].drop_duplicates().shape

(6236, 2)

## 10. Generalising the Quran Data Reading Process

The Arabic source showed us how the raw Quran files are structured and how verse records can be validated and converted into structured data.

The English and Indonesian sources follow the same `surah|ayah|text` format. Instead of repeating the same steps for each source, we can now generalise the reading process into a reusable function.

This is the same approach used in the [ChatQuran: A Multilingual RAG System](https://aviosit.com/projects/chatquran-multilingual-rag-system) project, where the reusable data-processing logic is moved from exploration into maintainable Python modules.

In [85]:
def read_tanzil_file(file_path: Path) -> pd.DataFrame:
    """Read a Tanzil text file into a pandas DataFrame."""
    records = []

    with file_path.open("r", encoding="utf-8") as file:
        for line_number, line in enumerate(file, start=1):
            line = line.rstrip("\n")

            if not line.strip() or line.startswith("#"):
                continue

            parts = line.split("|", maxsplit=2)

            if len(parts) != 3:
                raise ValueError(f"Invalid record at line {line_number}: {line!r}")

            surah, ayah, text = parts

            records.append({
                "surah": int(surah),
                "ayah": int(ayah),
                "text": text,
            })

    return pd.DataFrame(records, columns=["surah", "ayah", "text"])

In [86]:
arabic_df = read_tanzil_file(data_dir / "quran-uthmani.txt")
arabic_df.shape

(6236, 3)

In [87]:
arabic_df.equals(pd.DataFrame(records, columns=["surah", "ayah", "text"]))

True

## 11. Reading the Quran Sources

The same reading function can now be applied to the English and Indonesian sources.

Because the three sources follow the same record structure, we can use the same data-reading logic without repeating the earlier exploratory steps.

The English and Indonesian translations are also sourced through [Tanzil's translation collection](https://tanzil.net/trans/).

The specific translation sources and their licensing terms should be checked before redistributing or using the data outside this tutorial.

In [88]:
english_df = read_tanzil_file(data_dir / "en-saheeh-international.txt")
indonesian_df = read_tanzil_file(data_dir / "id-kemenag.txt")

In [89]:
{
    "Arabic": arabic_df.shape,
    "English": english_df.shape,
    "Indonesian": indonesian_df.shape,
}

{'Arabic': (6236, 3), 'English': (6236, 3), 'Indonesian': (6236, 3)}

In [90]:
arabic_keys = set(zip(arabic_df["surah"], arabic_df["ayah"]))
english_keys = set(zip(english_df["surah"], english_df["ayah"]))
indonesian_keys = set(zip(indonesian_df["surah"], indonesian_df["ayah"]))

arabic_keys == english_keys == indonesian_keys

True

## 12. Renaming the Text Columns

The three DataFrames currently use the same column names: `surah`, `ayah`, and `text`.

The `surah` and `ayah` columns identify the same Quran verse across all sources, while the `text` column contains language-specific content.

Before combining the sources, we rename the `text` column to make the language explicit.

In [91]:
arabic_df = arabic_df.rename(columns={"text": "arabic"})
english_df = english_df.rename(columns={"text": "english"})
indonesian_df = indonesian_df.rename(columns={"text": "indonesian"})

In [92]:
arabic_df.columns, english_df.columns, indonesian_df.columns

(Index(['surah', 'ayah', 'arabic'], dtype='str'),
 Index(['surah', 'ayah', 'english'], dtype='str'),
 Index(['surah', 'ayah', 'indonesian'], dtype='str'))

## 13. Combining the Quran Sources

The Quran sources contain the same `(surah, ayah)` keys, so they can be combined using this shared key.

We start by combining the Arabic and English DataFrames. The Indonesian translation will be added in the next step.

In [93]:
quran_df = arabic_df.merge(
    english_df,
    on=["surah", "ayah"],
    how="inner",
)

quran_df.head()

,surah,ayah,arabic,english
0,1,1,بِسْمِ ٱللَّهِ ٱلرَّحْمَـٰنِ ٱلرَّحِيمِ,"In the name of Allah, the Entirely Merciful, t..."
1,1,2,ٱلْحَمْدُ لِلَّهِ رَبِّ ٱلْعَـٰلَمِينَ,"[All] praise is [due] to Allah, Lord of the wo..."
2,1,3,ٱلرَّحْمَـٰنِ ٱلرَّحِيمِ,"The Entirely Merciful, the Especially Merciful,"
3,1,4,مَـٰلِكِ يَوْمِ ٱلدِّينِ,Sovereign of the Day of Recompense.
4,1,5,إِيَّاكَ نَعْبُدُ وَإِيَّاكَ نَسْتَعِينُ,It is You we worship and You we ask for help.


The first merge combines the Arabic and English DataFrames using `(surah, ayah)` as the join key.

Because both sources contain the same set of keys, the resulting DataFrame contains one Arabic text and one English translation for each Quran verse.

In [94]:
quran_df.shape

(6236, 4)

In [95]:
quran_df[["surah", "ayah"]].duplicated().sum()

np.int64(0)

## 14. Adding the Indonesian Translation

The Arabic and English sources are now aligned in a single DataFrame.

We can add the Indonesian translation using the same `(surah, ayah)` join key.

In [96]:
quran_df = quran_df.merge(
    indonesian_df,
    on=["surah", "ayah"],
    how="inner",
)

In [97]:
quran_df.head()

,surah,ayah,arabic,english,indonesian
0,1,1,بِسْمِ ٱللَّهِ ٱلرَّحْمَـٰنِ ٱلرَّحِيمِ,"In the name of Allah, the Entirely Merciful, t...",Dengan menyebut nama Allah Yang Maha Pemurah l...
1,1,2,ٱلْحَمْدُ لِلَّهِ رَبِّ ٱلْعَـٰلَمِينَ,"[All] praise is [due] to Allah, Lord of the wo...","Segala puji bagi Allah, Tuhan semesta alam."
2,1,3,ٱلرَّحْمَـٰنِ ٱلرَّحِيمِ,"The Entirely Merciful, the Especially Merciful,",Maha Pemurah lagi Maha Penyayang.
3,1,4,مَـٰلِكِ يَوْمِ ٱلدِّينِ,Sovereign of the Day of Recompense.,Yang menguasai di Hari Pembalasan.
4,1,5,إِيَّاكَ نَعْبُدُ وَإِيَّاكَ نَسْتَعِينُ,It is You we worship and You we ask for help.,"Hanya Engkaulah yang kami sembah, dan hanya ke..."


## 15. Validating the Prepared Dataset

The three Quran sources have now been combined into a single DataFrame.

Before saving the prepared dataset, we should validate its final structure, row count, key uniqueness, and missing values.

In [98]:
quran_df.shape

(6236, 5)

In [99]:
quran_df.columns

Index(['surah', 'ayah', 'arabic', 'english', 'indonesian'], dtype='str')

In [100]:
quran_df[["surah", "ayah"]].duplicated().sum()

np.int64(0)

In [101]:
quran_df.isna().sum()

surah         0
ayah          0
arabic        0
english       0
indonesian    0
dtype: int64

In [102]:
quran_df.dtypes

surah         int64
ayah          int64
arabic          str
english         str
indonesian      str
dtype: object

## 16. Saving the Prepared Dataset

The prepared Quran dataset has passed the final validation checks.

We can now save it as a Parquet file in the `processed` data directory.

Parquet is a columnar data format that preserves the tabular structure and data types of the DataFrame while providing efficient storage and loading.

In [103]:
processed_dir = Path("../data/processed")
processed_dir.mkdir(parents=True, exist_ok=True)

output_file = processed_dir / "quran.parquet"

In [104]:
quran_df.to_parquet(output_file, index=False)

## 17. Loading the Prepared Dataset

The prepared Quran dataset has been saved as a Parquet file.

We can now load the file again and verify that the stored dataset matches the DataFrame we prepared earlier.

This confirms that the dataset can be persisted and loaded again without changing its contents.

In [105]:
loaded_quran_df = pd.read_parquet(output_file)

loaded_quran_df.shape

(6236, 5)

In [106]:
loaded_quran_df.head()

,surah,ayah,arabic,english,indonesian
0,1,1,بِسْمِ ٱللَّهِ ٱلرَّحْمَـٰنِ ٱلرَّحِيمِ,"In the name of Allah, the Entirely Merciful, t...",Dengan menyebut nama Allah Yang Maha Pemurah l...
1,1,2,ٱلْحَمْدُ لِلَّهِ رَبِّ ٱلْعَـٰلَمِينَ,"[All] praise is [due] to Allah, Lord of the wo...","Segala puji bagi Allah, Tuhan semesta alam."
2,1,3,ٱلرَّحْمَـٰنِ ٱلرَّحِيمِ,"The Entirely Merciful, the Especially Merciful,",Maha Pemurah lagi Maha Penyayang.
3,1,4,مَـٰلِكِ يَوْمِ ٱلدِّينِ,Sovereign of the Day of Recompense.,Yang menguasai di Hari Pembalasan.
4,1,5,إِيَّاكَ نَعْبُدُ وَإِيَّاكَ نَسْتَعِينُ,It is You we worship and You we ask for help.,"Hanya Engkaulah yang kami sembah, dan hanya ke..."


In [107]:
loaded_quran_df.equals(quran_df)

True

## 18. From Exploration to ChatQuran

This notebook explored how the raw Quran sources can be inspected, validated, combined, and stored as a structured dataset.

The [ChatQuran: A Multilingual RAG System](https://aviosit.com/projects/chatquran-multilingual-rag-system) project turns these exploration steps into reusable Python code.

The main ideas from this notebook map directly to the project implementation:

- `read_tanzil_file()` reads the raw Tanzil files and validates their record structure.
- `(surah, ayah)` identifies each Quran verse and keeps the sources aligned.
- The text columns are renamed to preserve the language of each source.
- The Quran sources are merged into a single structured dataset.
- Validation checks help detect malformed records, duplicate keys, and missing data.
- The prepared dataset is stored as Parquet for later processing.

The notebook is intentionally exploratory, while the ChatQuran implementation moves the reusable logic into maintainable Python modules.

This tutorial is part of the [AviosIT Tutorials](https://aviosit.com/tutorials/preparing-quran-data-for-rag) collection by [AviosIT](https://aviosit.com).